# 02 — Validation checks + smoke test
1. Full checklist (validation checklist) → `results/checks/checks.md`
2. Every model: 1 fold, 1 seed, 2 epochs → `results/smoke/` (never mixed with real results)
3. Reproducibility: same config twice

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
%matplotlib inline
print("project root:", ROOT)

project root: <repo>


In [2]:
RUN_M4 = False   # True downloads facebook/wav2vec2-base (~360 MB) and runs M4 checks too
import pandas as pd
from src import checks, data, features as feat, train, utils
pd.set_option("display.max_colwidth", 200)
res = checks.run_all(with_m4=RUN_M4, overfit=True)
pd.DataFrame(res)

<repo>\src\train.py:327: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  losses.append(float(loss))
<venv>/site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Checks run 2026-11-21 17:43:25 (61 s)

| # | Check | Result | Detail |
|---|---|---|---|
| 1 | metrics (random): UAR==balanced_accuracy, macroF1==f1(macro), acc, mean(recalls)==UAR | PASS | /dUAR/=0.0e+00 /dF1/=0.0e+00 /dAcc/=0.0e+00 /dRec/=0.0e+00 |
| 2 | metrics (class never predicted): UAR==balanced_accuracy, macroF1==f1(macro), acc, mean(recalls)==UAR | PASS | /dUAR/=0.0e+00 /dF1/=0.0e+00 /dAcc/=0.0e+00 /dRec/=0.0e+00 |
| 3 | metrics (imbalanced): UAR==balanced_accuracy, macroF1==f1(macro), acc, mean(recalls)==UAR | PASS | /dUAR/=0.0e+00 /dF1/=0.0e+00 /dAcc/=0.0e+00 /dRec/=0.0e+00 |
| 4 | data: exactly 1440 files | PASS | n=1440 |
| 5 | data: neutral=96, other emotions=192 | PASS | {'neutral': 96, 'calm': 192, 'happy': 192, 'sad': 192, 'angry': 192, 'fearful': 192, 'disgust': 192, 'surprised': 192} |
| 6 | data: 60 files per actor (24 actors) | PASS | [60] |
| 7 | data: 12 F / 12 M actors | PASS | {'M': 12, 'F': 12} |
| 8 | data: label map fixed (index = code-1) and saved | PASS | 

,check,passed,detail
0,"metrics (random): UAR==balanced_accuracy, macroF1==f1(macro), acc, mean(recalls)==UAR",True,|dUAR|=0.0e+00 |dF1|=0.0e+00 |dAcc|=0.0e+00 |dRec|=0.0e+00
1,"metrics (class never predicted): UAR==balanced_accuracy, macroF1==f1(macro), acc, mean(recalls)==UAR",True,|dUAR|=0.0e+00 |dF1|=0.0e+00 |dAcc|=0.0e+00 |dRec|=0.0e+00
2,"metrics (imbalanced): UAR==balanced_accuracy, macroF1==f1(macro), acc, mean(recalls)==UAR",True,|dUAR|=0.0e+00 |dF1|=0.0e+00 |dAcc|=0.0e+00 |dRec|=0.0e+00
3,data: exactly 1440 files,True,n=1440
4,"data: neutral=96, other emotions=192",True,"{'neutral': 96, 'calm': 192, 'happy': 192, 'sad': 192, 'angry': 192, 'fearful': 192, 'disgust': 192, 'surprised': 192}"
5,data: 60 files per actor (24 actors),True,[60]
6,data: 12 F / 12 M actors,True,"{'M': 12, 'F': 12}"
7,data: label map fixed (index = code-1) and saved,True,"{'neutral': 0, 'calm': 1, 'happy': 2, 'sad': 3, 'angry': 4, 'fearful': 5, 'disgust': 6, 'surprised': 7}"
8,"folds: 6 folds x 4 actors, every actor exactly once",True,"[[4, 7, 14, 17], [11, 15, 18, 20], [1, 2, 21, 24], [6, 8, 9, 19], [3, 12, 16, 23], [5, 10, 13, 22]]"
9,folds: each fold 2 F + 2 M,True,"[['4F', '7M', '14F', '17M'], ['11M', '15M', '18F', '20F'], ['1M', '2F', '21M', '24F'], ['6F', '8F', '9M', '19M'], ['3M', '12F', '16F', '23M'], ['5M', '10F', '13M', '22F']]"


## Smoke runs (fold 0, seed 0, 2 epochs)

In [3]:
meta, splits = data.load_metadata(), data.load_splits()
cfgs = {m: utils.load_config(m) for m in utils.MODELS}
banks = {"logmel": feat.load_bank(cfgs["m1"], "logmel", meta), "mfcc": feat.load_bank(cfgs["m1"], "mfcc", meta)}
combos = [("m1", "logmel"), ("m1", "mfcc"), ("m2", "logmel"), ("m3", "logmel")]
if RUN_M4:
    banks["wave"] = feat.load_bank(cfgs["m4"], "wave", meta)
    combos.append(("m4", "wave"))
smoke = []
for m, f in combos:
    smoke.append(train.run_single(cfgs[m], f, "actor", 0, 0, meta, banks, splits, out_root="results/smoke",
                                  tag="smoke", max_epochs=2, save_checkpoint=False))
pd.DataFrame(smoke)[["run_id", "n_params", "epochs_run", "val_uar", "test_uar", "train_time_s"]]

== m1_logmel_actor_f0_s0_smoke | cuda | params 596,232 (trainable 596,232) | cfg e7e2115a1d
[m1_logmel_actor_f0_s0_smoke] ep   1 lr 1.00e-03 | train loss 1.990 acc 0.216 | val loss 1.932 UAR 0.242 *
[m1_logmel_actor_f0_s0_smoke] ep   2 lr 1.00e-03 | train loss 1.883 acc 0.302 | val loss 1.880 UAR 0.266 *
   best epoch 2 val UAR 0.2656 | TEST UAR 0.2930 acc 0.3125 F1 0.2433 | 8s
== m1_mfcc_actor_f0_s0_smoke | cuda | params 653,576 (trainable 653,576) | cfg e7e2115a1d
[m1_mfcc_actor_f0_s0_smoke] ep   1 lr 1.00e-03 | train loss 2.023 acc 0.190 | val loss 1.978 UAR 0.203 *
[m1_mfcc_actor_f0_s0_smoke] ep   2 lr 1.00e-03 | train loss 1.877 acc 0.300 | val loss 1.826 UAR 0.281 *
   best epoch 2 val UAR 0.2812 | TEST UAR 0.3398 acc 0.3625 F1 0.2845 | 7s
== m2_logmel_actor_f0_s0_smoke | cuda | params 1,276,904 (trainable 1,276,904) | cfg 1948a1fad9
[m2_logmel_actor_f0_s0_smoke] ep   1 lr 1.00e-03 | train loss 2.020 acc 0.198 | val loss 1.927 UAR 0.234 *
[m2_logmel_actor_f0_s0_smoke] ep   2 lr 1

,run_id,n_params,epochs_run,val_uar,test_uar,train_time_s
0,m1_logmel_actor_f0_s0_smoke,596232,2,0.265625,0.292969,7.9
1,m1_mfcc_actor_f0_s0_smoke,653576,2,0.281250,0.339844,7.5
2,m2_logmel_actor_f0_s0_smoke,1276904,2,0.296875,0.289062,3.9
3,m3_logmel_actor_f0_s0_smoke,2423048,2,0.316406,0.230469,1.5


In [4]:
# random-split protocol works too (M1, 2 epochs)
r = train.run_single(cfgs["m1"], "logmel", "random", None, 0, meta, banks, splits, out_root="results/smoke",
                     tag="smoke", max_epochs=2, save_checkpoint=False)
r["run_id"], r["test_uar"]

== m1_logmel_random_fR_s0_smoke | cuda | params 596,232 (trainable 596,232) | cfg e7e2115a1d
[m1_logmel_random_fR_s0_smoke] ep   1 lr 1.00e-03 | train loss 2.006 acc 0.208 | val loss 1.873 UAR 0.293 *
[m1_logmel_random_fR_s0_smoke] ep   2 lr 1.00e-03 | train loss 1.907 acc 0.270 | val loss 1.769 UAR 0.363 *
   best epoch 2 val UAR 0.3633 | TEST UAR 0.2812 acc 0.3000 F1 0.2494 | 7s


('m1_logmel_random_fR_s0_smoke', 0.28125)

## Reproducibility: same config + seed twice

In [5]:
a = train.run_single(cfgs["m1"], "logmel", "actor", 0, 0, meta, banks, splits, out_root="results/smoke",
                     tag="repro_a", max_epochs=3, save_checkpoint=False, verbose=False)
b = train.run_single(cfgs["m1"], "logmel", "actor", 0, 0, meta, banks, splits, out_root="results/smoke",
                     tag="repro_b", max_epochs=3, save_checkpoint=False, verbose=False)
ca = pd.read_csv(utils.RESULTS_DIR / "smoke/curves" / f"{a['run_id']}.csv")
cb = pd.read_csv(utils.RESULTS_DIR / "smoke/curves" / f"{b['run_id']}.csv")
diff = (ca[["train_loss", "val_loss", "val_uar"]] - cb[["train_loss", "val_loss", "val_uar"]]).abs().max()
rep = {"max_abs_diff": diff.to_dict(), "test_uar_a": a["test_uar"], "test_uar_b": b["test_uar"],
       "passed": bool(diff.max() < 1e-3 and abs(a["test_uar"] - b["test_uar"]) < 1e-3)}
utils.save_json(rep, utils.RESULTS_DIR / "checks" / "reproducibility.json")
rep

{'max_abs_diff': {'train_loss': 0.0, 'val_loss': 0.0, 'val_uar': 0.0},
 'test_uar_a': 0.3515625,
 'test_uar_b': 0.3515625,
 'passed': True}

## Next (terminal) — priority for tonight
```
python -m src.run_experiments --model m1 --folds 0-5 --seeds 0 --tag prelim
python -m src.run_experiments --model m3 --folds 0-5 --seeds 0 --tag prelim
python -m src.run_experiments --model m2 --folds 0-5 --seeds 0 --tag prelim
```